In [2]:
import mne
import numpy as np

from pathlib import Path

from scipy.signal import butter
from scipy.signal import filtfilt
from scipy.signal import iirnotch
from scipy.signal import welch

from scipy.stats import skew
from scipy.stats import kurtosis

In [3]:
pairs = [
    ('FC5','FC6'),
    ('FC3','FC4'),
    ('FC1','FC2'),

    ('C5','C6'),
    ('C3','C4'),
    ('C1','C2'),

    ('CP5','CP6'),
    ('CP3','CP4'),
    ('CP1','CP2'),

    ('FP1','FP2'),

    ('AF7','AF8'),
    ('AF3','AF4'),

    ('F7','F8'),
    ('F5','F6'),
    ('F3','F4'),
    ('F1','F2'),

    ('FT7','FT8'),

    ('T7','T8'),
    ('T9','T10'),

    ('TP7','TP8'),

    ('P7','P8'),
    ('P5','P6'),
    ('P3','P4'),
    ('P1','P2'),

    ('PO7','PO8'),
    ('PO3','PO4'),

    ('O1','O2')
]

print("Number of pairs =", len(pairs))

Number of pairs = 27


In [4]:
def create_differential_channels(raw, pairs):

    # Clean channel names
    clean_names = {
        ch: ch.replace(".", "").upper()
        for ch in raw.ch_names
    }

    raw.rename_channels(clean_names)

    # Get EEG data
    data = raw.get_data()

    # Create fresh channel index dictionary
    ch_idx = {
        ch: i
        for i, ch in enumerate(raw.ch_names)
    }

    diff_data = []

    for left, right in pairs:

        left_idx = ch_idx[left]
        right_idx = ch_idx[right]

        diff_signal = data[left_idx] - data[right_idx]

        diff_data.append(diff_signal)

    diff_data = np.array(diff_data)

    diff_names = [
        f"{left}-{right}"
        for left, right in pairs
    ]

    return diff_data, diff_names

In [5]:
def apply_notch_filter(data, fs=160, freq=50, Q=30):

    b, a = iirnotch(
        w0=freq,
        Q=Q,
        fs=fs
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=1
    )

    return filtered

In [6]:
def apply_bandpass_filter(
    data,
    fs=160,
    lowcut=0.5,
    highcut=70,
    order=4
):

    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=1
    )

    return filtered

In [7]:
def extract_11_features(signal, fs=160):

    features = []

    # 1 Mean
    features.append(np.mean(signal))

    # 2 Variance
    features.append(np.var(signal))

    # 3 Skewness
    features.append(skew(signal))

    # 4 Kurtosis
    features.append(kurtosis(signal))

    # 5 Zero Crossing Count
    zc = np.sum(
        np.diff(
            np.sign(signal)
        ) != 0
    )

    features.append(zc)

    # 6 Area
    features.append(
        np.trapezoid(np.abs(signal))
    )

    # 7 Range
    features.append(
        np.max(signal) - np.min(signal)
    )

    # PSD
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=len(signal)
    )

    # 8 Delta
    delta = np.sum(
        psd[
            (freqs >= 0.5) &
            (freqs < 4)
        ]
    )

    # 9 Theta
    theta = np.sum(
        psd[
            (freqs >= 4) &
            (freqs < 8)
        ]
    )

    # 10 Alpha
    alpha = np.sum(
        psd[
            (freqs >= 8) &
            (freqs < 12)
        ]
    )

    # 11 Beta
    beta = np.sum(
        psd[
            (freqs >= 12) &
            (freqs < 30)
        ]
    )

    features.extend([
        delta,
        theta,
        alpha,
        beta
    ])

    return np.array(features)

In [8]:
def process_edf_file(edf_path, offset_samples=0):

    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False
    )

    events, event_id = mne.events_from_annotations(
        raw,
        verbose=False
    )

    movement_events = events[
        np.isin(
            events[:,2],
            [event_id["T1"], event_id["T2"]]
        )
    ]

    diff_data, _ = create_differential_channels(
        raw,
        pairs
    )

    diff_data = apply_notch_filter(diff_data)

    diff_data = apply_bandpass_filter(diff_data)

    segment_length = 320

    segments = []
    labels = []

    for event in movement_events:

        start = event[0] + offset_samples
        end = start + segment_length

        if end <= diff_data.shape[1]:

            segments.append(
                diff_data[:,start:end]
            )

            labels.append(event[2])

    segments = np.array(segments)

    labels = np.where(
        np.array(labels)==event_id["T1"],
        0,
        1
    )

    return segments, labels

In [10]:
offsets = {

    "0 ms":0,

    "250 ms":40,

    "500 ms":80,

    "750 ms":120,

    "1000 ms":160

}

for name,offset in offsets.items():

    segments,labels = process_edf_file(

        "../data/eegmmidb/files/S001/S001R04.edf",

        offset_samples=offset

    )

    print(name)

    print(segments.shape)

    print(labels.shape)

    print()

0 ms
(15, 27, 320)
(15,)

250 ms
(15, 27, 320)
(15,)

500 ms
(15, 27, 320)
(15,)

750 ms
(15, 27, 320)
(15,)

1000 ms
(15, 27, 320)
(15,)



In [11]:
# ==================================================
# Segment Offset
# ==================================================

# 0 samples   = 0 ms
# 40 samples  = 250 ms
# 80 samples  = 500 ms
# 120 samples = 750 ms
# 160 samples = 1000 ms

OFFSET_SAMPLES = 80

print(f"Segment Offset = {OFFSET_SAMPLES} samples")
print(f"Time Offset = {OFFSET_SAMPLES/160:.2f} seconds")

Segment Offset = 80 samples
Time Offset = 0.50 seconds


In [12]:
def process_edf_file(edf_path, offset_samples=0):

    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False
    )

    events, event_id = mne.events_from_annotations(
        raw,
        verbose=False
    )

    movement_events = events[
        np.isin(
            events[:,2],
            [event_id["T1"], event_id["T2"]]
        )
    ]

    diff_data, _ = create_differential_channels(
        raw,
        pairs
    )

    diff_data = apply_notch_filter(diff_data)

    diff_data = apply_bandpass_filter(diff_data)

    segment_length = 320

    segments = []
    labels = []

    for event in movement_events:

        start = event[0] + OFFSET_SAMPLES
        end = start + segment_length

        if end <= diff_data.shape[1]:

            segments.append(
                diff_data[:,start:end]
            )

            labels.append(event[2])

    segments = np.array(segments)

    labels = np.where(
        np.array(labels)==event_id["T1"],
        0,
        1
    )

    return segments, labels

In [13]:
excluded_subjects = [
    43,
    88,
    89,
    92,
    100,
    104
]

valid_subjects = []

for s in range(1,110):

    if s not in excluded_subjects:
        valid_subjects.append(s)

print(len(valid_subjects))

103


In [14]:
all_X = []
all_y = []
all_subjects = []

for subject in valid_subjects:

    subject_id = f"S{subject:03d}"

    for run in ["R04","R08","R12"]:

        edf_path = Path(
            f"../data/eegmmidb/files/{subject_id}/{subject_id}{run}.edf"
        )

        if not edf_path.exists():
            continue

        X_file, y_file = process_edf_file(
            edf_path
        )

        all_X.append(X_file)
        all_y.append(y_file)

        all_subjects.extend(
            [subject] * len(y_file)
        )

        print(
            f"{subject_id}{run} -> {X_file.shape}"
        )

S001R04 -> (15, 27, 320)
S001R08 -> (15, 27, 320)
S001R12 -> (15, 27, 320)
S002R04 -> (15, 27, 320)
S002R08 -> (15, 27, 320)
S002R12 -> (15, 27, 320)
S003R04 -> (15, 27, 320)
S003R08 -> (15, 27, 320)
S003R12 -> (15, 27, 320)
S004R04 -> (15, 27, 320)
S004R08 -> (15, 27, 320)
S004R12 -> (15, 27, 320)
S005R04 -> (15, 27, 320)
S005R08 -> (15, 27, 320)
S005R12 -> (15, 27, 320)
S006R04 -> (15, 27, 320)
S006R08 -> (15, 27, 320)
S006R12 -> (15, 27, 320)
S007R04 -> (15, 27, 320)
S007R08 -> (15, 27, 320)
S007R12 -> (15, 27, 320)
S008R04 -> (15, 27, 320)
S008R08 -> (15, 27, 320)
S008R12 -> (15, 27, 320)
S009R04 -> (15, 27, 320)
S009R08 -> (15, 27, 320)
S009R12 -> (15, 27, 320)
S010R04 -> (15, 27, 320)
S010R08 -> (15, 27, 320)
S010R12 -> (15, 27, 320)
S011R04 -> (15, 27, 320)
S011R08 -> (15, 27, 320)
S011R12 -> (15, 27, 320)
S012R04 -> (15, 27, 320)
S012R08 -> (15, 27, 320)
S012R12 -> (15, 27, 320)
S013R04 -> (15, 27, 320)
S013R08 -> (15, 27, 320)
S013R12 -> (15, 27, 320)
S014R04 -> (15, 27, 320)


In [16]:
X = np.concatenate(
    all_X,
    axis=0
)

y = np.concatenate(
    all_y,
    axis=0
)

subjects = np.array(
    all_subjects
)

print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 27, 320)
(4635,)
(4635,)


In [17]:
window_size = 80
step_size = 40

all_windows = []

for segment in X:

    segment_windows = []

    for start in range(
        0,
        320 - window_size + 1,
        step_size
    ):

        end = start + window_size

        window = segment[:, start:end]

        segment_windows.append(window)

    all_windows.append(segment_windows)

all_windows = np.array(all_windows)

print(all_windows.shape)

(4635, 7, 27, 80)


In [18]:
all_features = []

for trial in all_windows:

    trial_features = []

    for window in trial:

        window_features = []

        for channel in window:

            feats = extract_11_features(channel)

            window_features.extend(feats)

        trial_features.append(window_features)

    all_features.append(trial_features)

all_features = np.array(all_features)

print(all_features.shape)

(4635, 7, 297)


In [19]:
np.save(
    "../processed/X_offset80.npy",
    all_features
)

np.save(
    "../processed/y_offset80.npy",
    y
)

np.save(
    "../processed/subjects_offset80.npy",
    subjects
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [20]:
X_new = np.load("../processed/X_offset80.npy")
y_new = np.load("../processed/y_offset80.npy")
subjects_new = np.load("../processed/subjects_offset80.npy")

print(X_new.shape)
print(y_new.shape)
print(subjects_new.shape)

print("Unique Subjects:", len(np.unique(subjects_new)))

(4635, 7, 297)
(4635,)
(4635,)
Unique Subjects: 103
